# NB12 - Capstone: Private Federated Learning Pipeline
## Background
This capstone assembles the complete private FL pipeline: DP-SGD for client-side privacy, Byzantine-robust aggregation for server-side security, and a privacy audit at the end. This is the architecture used in production FL systems at companies like Apple, Google, and hospitals for cross-silo federated learning.

## What You'll Learn
- Full pipeline: DP-SGD training + Byzantine-robust aggregation + privacy audit
- Privacy budget tracking across rounds
- Security monitoring: detecting poisoned client updates
- Privacy-utility-communication tradeoff surface

## Why This Matters
The complete private FL pipeline is the gold standard for privacy-preserving ML in regulated industries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

np.random.seed(42)

# ── Private FL components ──────────────────────────────────────────────────
@dataclass
class DPConfig:
    noise_multiplier: float = 1.0
    max_grad_norm: float = 1.0
    delta: float = 1e-5

class PrivateFLClient:
    def __init__(self, client_id, X, y, n_classes, n_features, dp_config: DPConfig,
                 is_byzantine=False):
        self.id = client_id
        self.X, self.y = X, y
        self.n_classes = n_classes
        self.n_features = n_features
        self.dp = dp_config
        self.is_byzantine = is_byzantine
        self.W = np.zeros((n_features, n_classes))
        self.b = np.zeros(n_classes)
        self.privacy_spent = 0.0

    def local_step(self, W_global, b_global, lr=0.05) -> Tuple[np.ndarray, np.ndarray, float]:
        """One step of DP-SGD, return (grad_W, grad_b, privacy_cost)."""
        self.W = W_global.copy()
        self.b = b_global.copy()

        if self.is_byzantine:
            # Byzantine client sends random noise
            noise_W = np.random.randn(*W_global.shape) * 10
            noise_b = np.random.randn(*b_global.shape) * 10
            return noise_W, noise_b, 0.0

        n = len(self.X)
        h = self.X @ self.W + self.b
        h -= h.max(axis=1, keepdims=True)
        e = np.exp(h); probs = e / e.sum(axis=1, keepdims=True)
        dz = probs.copy(); dz[np.arange(n), self.y] -= 1

        # Per-sample gradients for clipping
        grad_W_samples = np.einsum('ni,nj->nij', self.X, dz)  # (n, d_in, n_cls)
        grad_b_samples = dz  # (n, n_cls)

        # Clip per-sample gradients
        for i in range(n):
            norm = (np.linalg.norm(grad_W_samples[i])**2 +
                    np.linalg.norm(grad_b_samples[i])**2)**0.5
            scale = min(1.0, self.dp.max_grad_norm / (norm + 1e-10))
            grad_W_samples[i] *= scale
            grad_b_samples[i] *= scale

        # Aggregate and add noise
        grad_W = grad_W_samples.mean(axis=0)
        grad_b = grad_b_samples.mean(axis=0)
        noise_W = np.random.randn(*grad_W.shape) * self.dp.noise_multiplier * \
                  self.dp.max_grad_norm / n
        noise_b = np.random.randn(*grad_b.shape) * self.dp.noise_multiplier * \
                  self.dp.max_grad_norm / n

        # Privacy accounting (RDP approximation)
        q = min(1.0, 32 / n)  # sampling rate
        eps_step = q**2 / (2 * self.dp.noise_multiplier**2)
        self.privacy_spent += eps_step

        return grad_W + noise_W, grad_b + noise_b, eps_step

def trimmed_mean(updates: List[np.ndarray], trim_frac=0.2) -> np.ndarray:
    """Byzantine-robust trimmed mean aggregation."""
    stack = np.array([u.flatten() for u in updates])
    n = len(stack)
    k = max(1, int(n * trim_frac))
    sorted_stack = np.sort(stack, axis=0)
    trimmed = sorted_stack[k:-k] if k > 0 else sorted_stack
    return trimmed.mean(axis=0).reshape(updates[0].shape)

print('Private FL components initialized.')

In [ ]:
# ── Simulate the full private FL pipeline ─────────────────────────────────
n_features, n_classes = 6, 3
n_clients = 8
n_byzantine = 2

# Dataset
X_all, y_all = [], []
class_means = np.eye(n_classes, n_features) * 3
for c in range(n_classes):
    X_all.append(class_means[c] + np.random.randn(100, n_features))
    y_all.extend([c]*100)
X_all, y_all = np.vstack(X_all), np.array(y_all)

X_test, y_test = [], []
for c in range(n_classes):
    X_test.append(class_means[c] + np.random.randn(30, n_features))
    y_test.extend([c]*30)
X_test, y_test = np.vstack(X_test), np.array(y_test)

dp_config = DPConfig(noise_multiplier=0.5, max_grad_norm=1.0)
clients = []
n_per_client = len(X_all) // n_clients
for i in range(n_clients):
    is_byz = (i < n_byzantine)
    Xi = X_all[i*n_per_client:(i+1)*n_per_client]
    yi = y_all[i*n_per_client:(i+1)*n_per_client]
    clients.append(PrivateFLClient(i, Xi, yi, n_classes, n_features,
                                    dp_config, is_byzantine=is_byz))

# Global model
W_global = np.zeros((n_features, n_classes))
b_global = np.zeros(n_classes)

n_rounds = 15
accs, total_eps = [], 0.0

for r in range(n_rounds):
    grad_Ws, grad_bs = [], []
    round_eps = 0.0
    for client in clients:
        gW, gb, eps = client.local_step(W_global, b_global)
        grad_Ws.append(gW); grad_bs.append(gb)
        round_eps = max(round_eps, eps)

    # Byzantine-robust aggregation
    agg_W = trimmed_mean(grad_Ws, trim_frac=0.25)
    agg_b = trimmed_mean(grad_bs, trim_frac=0.25)

    W_global = W_global - 0.1 * agg_W
    b_global = b_global - 0.1 * agg_b
    total_eps += round_eps

    h = X_test @ W_global + b_global
    acc = (np.argmax(h, axis=1) == y_test).mean()
    accs.append(acc)
    print(f'Round {r+1:2d}: acc={acc:.3f}, round_eps={round_eps:.4f}, total_eps={total_eps:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, n_rounds+1), accs, '-o', linewidth=2)
axes[0].set_title(f'Private FL: {n_byzantine}/{n_clients} Byzantine clients')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('Test accuracy')
axes[0].grid(True, alpha=0.3)

cumulative_eps = np.cumsum([total_eps/n_rounds]*n_rounds)
axes[1].plot(range(1, n_rounds+1), cumulative_eps, 'r-o', linewidth=2)
axes[1].set_title('Cumulative Privacy Budget')
axes[1].set_xlabel('Round'); axes[1].set_ylabel('Cumulative epsilon (RDP)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s16_12_private_fl_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

print()
print('=== Series 16: Privacy-Preserving ML & Federated Learning Complete ===')